# Events And Tracks

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

An entry of a "Hits" tree is one energy deposit. Several deposits belong to one
track, several tracks to one event, several events to one run — and which
entries belong together is written in three identifier branches, `runID`,
`eventID` and `trackID`, exactly as GATE wrote them.

The one thing to get right is that GATE numbers events within a run and tracks
within an event. This notebook reads a file holding three runs, shows what
goes wrong when an event is named by `eventID` alone, selects runs and events
the way the package intends, counts events and tracks by the right keys, and
numbers the hits of every track in the order they happened.

## What will you learn from this notebook

- why an event is the pair `(runID, eventID)` and a track the triple
  `(runID, eventID, trackID)`
- how to select the rows of a run and of an event
- how to count events and tracks without counting too few
- why the package never renumbers the identifiers, and how to make a number of
  your own when an analysis needs one
- how to number the hits of every track and keep the first two

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The file holds three runs of one simulation, one tree per run, each cut to its
first 500 hits; it is described in [`../data/README.md`](../data/README.md).
How such a file is read as one dataset is the subject of tutorial 06 — here it
is one call:

In [2]:
import pandas as pd

from opengate_gate_tree import read_hits_trees

hits = read_hits_trees(
    paths.input_example("hits-one-tree-per-run.root"),
    ["runID", "eventID", "trackID", "time", "edep", "processName"],
    add_source_branch=False,
)
frame = hits.to_dataframe()
frame.head(8)

,runID,eventID,trackID,time,edep,processName
0,0,0,2,0.000002,0.511000,PhotoElectric
1,0,0,1,0.000002,0.011562,Compton
2,0,0,1,0.000002,0.499438,PhotoElectric
3,0,1,2,0.000007,0.511000,PhotoElectric
4,0,1,1,0.000007,0.193163,Compton
5,0,1,1,0.000007,0.317837,PhotoElectric
6,0,2,2,0.000009,0.511000,PhotoElectric
7,0,2,1,0.000009,0.511000,PhotoElectric


The first eight hits: event 0 of run 0 left three deposits, one from track 2
and two from track 1; event 1 left three as well; event 2 two. A row is a
deposit, not a particle.

### The Identifiers Start Again In Every Run

Where `eventID` starts and ends in each run, and where the time does:

In [3]:
frame.groupby("runID").agg(
    first_event=("eventID", "min"),
    last_event=("eventID", "max"),
    first_time=("time", "min"),
    last_time=("time", "max"),
)

,first_event,last_event,first_time,last_time
runID,,,,
0,0,149,0.000002,0.001545
1,0,153,0.020001,0.021625
2,0,153,0.040004,0.041716


Every run numbers its events from 0, while the time — in seconds, as GATE
writes it — keeps running: the hits of run 1 begin at 0.02 s, those of run 2 at
0.04 s. The last values are where the trees were cut, not where the runs ended.

So `eventID` alone names three different things here. Asking for event 5 by
its identifier alone:

In [4]:
frame[frame["eventID"] == 5]

,runID,eventID,trackID,time,edep,processName
16,0,5,2,0.000025,0.511000,PhotoElectric
17,0,5,1,0.000025,0.511000,PhotoElectric
516,1,5,2,0.020055,0.511000,PhotoElectric
517,1,5,1,0.020055,0.511000,PhotoElectric
1019,2,5,2,0.040084,0.107709,Compton
1020,2,5,2,0.040084,0.403291,PhotoElectric
1021,2,5,1,0.040084,0.207644,Compton
1022,2,5,1,0.040084,0.124949,Compton
1023,2,5,1,0.040084,0.178407,PhotoElectric


Nine deposits from three decays, 20 ms apart. Run 0 and run 1 each saw two
gammas deposit 511 keV, while run 2 saw two tracks scatter and leave five
deposits. Treated as one event, they would make a decay that never happened.

### Selecting A Run And An Event

The package selects an event by both identifiers, and offers no filter that
takes `eventID` alone: `frame["eventID"] == 5` is one comparison, and it should
look like the guess it is. The filters live in the `gate` namespace, which
importing the package registers on every frame:

In [5]:
frame.gate.by_event(2, 5)

,runID,eventID,trackID,time,edep,processName
1019,2,5,2,0.040084,0.107709,Compton
1020,2,5,2,0.040084,0.403291,PhotoElectric
1021,2,5,1,0.040084,0.207644,Compton
1022,2,5,1,0.040084,0.124949,Compton
1023,2,5,1,0.040084,0.178407,PhotoElectric


The same selection as a mask, which composes with any other condition on the
same frame — here, the deposits of that event above 150 keV:

In [6]:
of_the_event = frame.gate.is_from_event(2, 5)
frame[of_the_event & (frame["edep"] > 0.15)]

,runID,eventID,trackID,time,edep,processName
1020,2,5,2,0.040084,0.403291,PhotoElectric
1021,2,5,1,0.040084,0.207644,Compton
1023,2,5,1,0.040084,0.178407,PhotoElectric


A whole run is `by_run`, and `is_from_run` gives its mask:

In [7]:
second_run = frame.gate.by_run(1)
len(second_run), sorted(second_run["runID"].unique().tolist())

(500, [1])

### Counting Events And Tracks

Counting distinct values of the wrong key is the quiet version of the same
mistake: nothing fails, the number is just too small. Events by `eventID`
alone, and by the pair:

In [8]:
events_by_id = frame["eventID"].nunique()
events_by_pair = len(frame[["runID", "eventID"]].drop_duplicates())
events_by_id, events_by_pair

(154, 458)

154 against 458: counted by the identifier alone, two of every three events
disappear. Tracks are numbered within an event in the same way — every event
has a track 1 — so a track is the triple:

In [9]:
tracks_without_run = len(frame[["eventID", "trackID"]].drop_duplicates())
tracks = len(frame[["runID", "eventID", "trackID"]].drop_duplicates())
tracks_without_run, tracks

(308, 915)

915 tracks, not 308. The statistics report the package writes counts by
exactly these keys, and says in the report which branches it counted by — see
the guide on statistics.

### Why Nothing Is Renumbered

Giving every event a number of its own would make the identifiers unique, and
the package deliberately does not do it. A simulation being debugged is run
twice with the same seed, and the two outputs are compared event by event to
see what a change did; a number derived from the contents of a file would stop
matching between the two as soon as anything else about them differed. The
identifiers also tie a row back to the GATE output and the logs of the
simulation that produced it.

When an analysis does want one number per event — as the index of an array,
say — it makes one for itself, in its own column, knowing that the number
belongs to this frame and to nothing else:

In [10]:
frame["eventNumber"] = frame.groupby(["runID", "eventID"]).ngroup()
frame["eventNumber"].nunique(), int(frame["eventNumber"].max())

(458, 457)

458 numbers, from 0 to 457, one per pair. Read the same events from a file cut
differently and they would be numbered differently — which is exactly why the
package leaves this to the analysis.

### The Hits Of A Track, In Order

How many deposits a track leaves:

In [11]:
track_key = ["runID", "eventID", "trackID"]

frame.groupby(track_key).size().value_counts().sort_index()

1    486
2    299
3    106
4     22
5      2
Name: count, dtype: int64

Most tracks leave one deposit, and a few leave up to five. To number the
deposits of every track in the order they happened, the rows are sorted by the
track and then by time — the sort is stable, so deposits at the same time keep
the order GATE wrote them in — and `cumcount` counts within each track:

In [12]:
ordered = frame.sort_values([*track_key, "time"], kind="stable")
ordered["hitIndex"] = ordered.groupby(track_key).cumcount()
ordered["hitsInTrack"] = ordered.groupby(track_key)["edep"].transform("size")

ordered.gate.by_event(2, 5)

,runID,eventID,trackID,time,edep,processName,eventNumber,hitIndex,hitsInTrack
1021,2,5,1,0.040084,0.207644,Compton,309,0,3
1022,2,5,1,0.040084,0.124949,Compton,309,1,3
1023,2,5,1,0.040084,0.178407,PhotoElectric,309,2,3
1019,2,5,2,0.040084,0.107709,Compton,309,0,2
1020,2,5,2,0.040084,0.403291,PhotoElectric,309,1,2


The first two hits of every track that has two — the pair a Compton analysis
needs, since the first deposit and the second give the direction the scattered
gamma flew in:

In [13]:
first_two = ordered[(ordered["hitsInTrack"] >= 2) & (ordered["hitIndex"] < 2)]
len(first_two), first_two.groupby(track_key).ngroups

(858, 429)

429 tracks, two rows each. What happened at each position along a track is a
check that the numbering follows the physics:

In [14]:
multi_hit = ordered[ordered["hitsInTrack"] >= 2]
pd.crosstab(multi_hit["hitIndex"], multi_hit["processName"])

processName,Compton,PhotoElectric
hitIndex,,
0,429,0
1,130,299
2,24,106
3,2,22
4,0,2


In [15]:
last_hits = ordered[ordered["hitIndex"] == ordered["hitsInTrack"] - 1]
last_hits["processName"].value_counts()

processName
PhotoElectric    915
Name: count, dtype: int64

Every track with more than one deposit begins with a Compton scattering, and
every track in this file — those with one deposit included — ends with a
photoelectric absorption, which is the deposit that stops a gamma. A numbering
that had mixed up the tracks of different runs would not give tables this
clean.

## Summary

An event is named by `runID` and `eventID` together and a track by those two
and `trackID`, because GATE numbers each within the level above it; `by_run`,
`by_event` and `is_from_event` select by them, and every count and grouping has
to use the whole key. The package never renumbers the identifiers, so a number
of one's own is made with `groupby(...).ngroup()`, and the hits of a track are
numbered in order with `groupby(...).cumcount()`.

## Bibliography

- [Event Identifiers](https://opengate-gate-tree.readthedocs.io/en/latest/guide/events.html)
  — why an event is a run and an event together, and why identifiers are never
  renumbered
- [Filtering](https://opengate-gate-tree.readthedocs.io/en/latest/guide/filtering.html)
  — runs and events, and the `gate` namespace
- [Merging Trees](https://opengate-gate-tree.readthedocs.io/en/latest/guide/merging.html)
  — reading a file split per run as one dataset
- [Statistics](https://opengate-gate-tree.readthedocs.io/en/latest/guide/statistics.html)
  — the keys events and tracks are counted by in a report
- [Units](https://opengate-gate-tree.readthedocs.io/en/latest/guide/units.html)
  — why `time` is in seconds and `edep` in MeV

## Changelog

- v1: written.